# Cardamom Pod Grading — Fixed Training (Colab)

Use this notebook instead of `Cardamom_pod_grading (2).ipynb`.

## What was wrong before
1. **Crop padding = 0.8** made almost every pod fill the frame → model leaned on size/composition, not colour/damage.
2. **Strong RandomBrightness/Contrast** washed out green vs brown (the main Grade A/B/C signal).
3. **Failed crops used the full original image** → mixed distributions in training.
4. MobileNetV2 + no label smoothing → overconfident wrong grades (e.g. Grade C → Grade A).

## What this version changes
- Smaller crop padding (`CROP_PAD_RATIO = 0.25`) so colour/blemishes stay informative
- Mild geometric augmentation only (colour-preserving)
- Skip failed crops (no raw fallback)
- EfficientNetB0 + label smoothing
- Checkpoint on `val_loss` (better calibration)
- Saves `best_cardamom_grade_fixed.keras` + `class_names.json`

## After training — important
Update the same pad in `main.py` → `detect_pod_crop_and_measure`:

```python
pad = int(max(w, h) * 0.25) + 20   # was 0.8
```

Train and serve must use the **same** crop.


In [ ]:
# 1) Setup
import os
import json
import shutil
import random
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
CROP_PAD_RATIO = 0.25   # MUST match server after deploy (was 0.8)
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


In [ ]:
# 2) Mount Drive + unzip dataset
from google.colab import drive
drive.mount('/content/drive')

# Change this if your zip path is different
ZIP_PATH = "/content/drive/MyDrive/cardamom_grade_dataset.zip"

WORK_DIR = Path("/content/cardamom_grading")
RAW_DIR = WORK_DIR / "raw"

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
RAW_DIR.mkdir(parents=True, exist_ok=True)

!unzip -q "{ZIP_PATH}" -d "{RAW_DIR}"

print("Unzipped folders:")
!find "{RAW_DIR}" -maxdepth 4 -type d | sort


In [ ]:
# 3) Find dataset root (train / validation / test)
candidates = []
for p in RAW_DIR.rglob("*"):
    if (p / "train").is_dir() and (p / "validation").is_dir() and (p / "test").is_dir():
        candidates.append(p)

if not candidates:
    raise RuntimeError(
        "Could not find dataset root. Zip must contain train, validation, and test folders."
    )

DATASET_ROOT = candidates[0]
print("Dataset root:", DATASET_ROOT)

for split in ["train", "validation", "test"]:
    print("\n", split.upper())
    split_path = DATASET_ROOT / split
    for class_dir in sorted(split_path.iterdir()):
        if class_dir.is_dir():
            count = len([f for f in class_dir.iterdir() if f.suffix.lower() in IMAGE_EXTS])
            print(f"  {class_dir.name}: {count}")


In [ ]:
# 4) Crop helper (colour-safe framing)
def crop_pod_to_square(img_bgr, pad_ratio=CROP_PAD_RATIO):
    """Detect pod on light background, crop with modest padding, square canvas."""
    if img_bgr is None:
        return None

    h_img, w_img = img_bgr.shape[:2]
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s, v = cv2.split(hsv)

    mask = ((s > 25) & (v < 245)).astype(np.uint8) * 255
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    contour = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(contour)
    min_area = max(200, h_img * w_img * 0.0002)
    if area < min_area:
        return None

    x, y, w, h = cv2.boundingRect(contour)
    pad = int(max(w, h) * pad_ratio) + 20

    x1 = max(0, x - pad)
    y1 = max(0, y - pad)
    x2 = min(w_img, x + w + pad)
    y2 = min(h_img, y + h + pad)

    crop = img_bgr[y1:y2, x1:x2]
    if crop.size == 0:
        return None

    ch, cw = crop.shape[:2]
    side = max(ch, cw)
    canvas = np.ones((side, side, 3), dtype=np.uint8) * 255
    y_offset = (side - ch) // 2
    x_offset = (side - cw) // 2
    canvas[y_offset:y_offset + ch, x_offset:x_offset + cw] = crop
    return canvas


In [ ]:
# 5) Build cropped dataset (SKIP failed crops — do not use raw fallback)
CROPPED_ROOT = WORK_DIR / "dataset_cropped"
if CROPPED_ROOT.exists():
    shutil.rmtree(CROPPED_ROOT)

failed_images = []
saved_count = 0

for split in ["train", "validation", "test"]:
    split_input = DATASET_ROOT / split
    for class_dir in sorted(split_input.iterdir()):
        if not class_dir.is_dir():
            continue
        out_class_dir = CROPPED_ROOT / split / class_dir.name
        out_class_dir.mkdir(parents=True, exist_ok=True)

        for img_path in class_dir.iterdir():
            if img_path.suffix.lower() not in IMAGE_EXTS:
                continue
            img_bgr = cv2.imread(str(img_path))
            cropped = crop_pod_to_square(img_bgr)
            if cropped is None:
                failed_images.append(str(img_path))
                continue  # skip — do NOT save the uncropped original
            cv2.imwrite(str(out_class_dir / img_path.name), cropped)
            saved_count += 1

print("Cropped dataset:", CROPPED_ROOT)
print("Saved images:", saved_count)
print("Skipped (failed crop):", len(failed_images))
if failed_images[:5]:
    print("Examples skipped:", *failed_images[:5], sep="\n  ")

for split in ["train", "validation", "test"]:
    print("\n", split.upper())
    for class_dir in sorted((CROPPED_ROOT / split).iterdir()):
        if class_dir.is_dir():
            count = len([f for f in class_dir.iterdir() if f.suffix.lower() in IMAGE_EXTS])
            print(f"  {class_dir.name}: {count}")


In [ ]:
# 6) Preview samples
def show_random_samples(root, split="train", n=9):
    split_path = Path(root) / split
    all_images = []
    for class_dir in sorted(split_path.iterdir()):
        if class_dir.is_dir():
            for img_path in class_dir.iterdir():
                if img_path.suffix.lower() in IMAGE_EXTS:
                    all_images.append((img_path, class_dir.name))

    samples = random.sample(all_images, min(n, len(all_images)))
    plt.figure(figsize=(10, 10))
    for i, (img_path, label) in enumerate(samples):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        plt.subplot(3, 3, i + 1)
        plt.imshow(img)
        plt.title(label)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

show_random_samples(CROPPED_ROOT, "train", 9)


In [ ]:
# 7) Datasets
train_ds = tf.keras.utils.image_dataset_from_directory(
    CROPPED_ROOT / "train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True,
    seed=SEED,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    CROPPED_ROOT / "validation",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    CROPPED_ROOT / "test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,
)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Class names:", class_names)

# Keep class order identical across splits
assert val_ds.class_names == class_names, "validation class order mismatch"
assert test_ds.class_names == class_names, "test class order mismatch"

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)


In [ ]:
# 8) Class weights (balance A / B / C)
train_counts = Counter()
for i, class_name in enumerate(class_names):
    class_path = CROPPED_ROOT / "train" / class_name
    train_counts[class_name] = len(
        [f for f in class_path.iterdir() if f.suffix.lower() in IMAGE_EXTS]
    )

print("Train counts:", dict(train_counts))
total = sum(train_counts.values())
class_weight = {
    i: total / (num_classes * train_counts[class_names[i]])
    for i in range(num_classes)
}
print("Class weights:", class_weight)


In [ ]:
# 9) Model — EfficientNetB0 + mild colour-preserving augmentation
# Avoid strong brightness/contrast: those destroy green vs brown cues.

data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.12),
        layers.RandomZoom(0.08),
        layers.RandomTranslation(0.08, 0.08),
        layers.RandomContrast(0.08),  # mild only
    ],
    name="data_augmentation",
)

base_model = EfficientNetB0(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
# Direct call serializes cleanly (avoid Lambda + custom_objects on load)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.40)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = keras.Model(inputs, outputs, name="cardamom_pod_grader")

# SparseCategoricalCrossentropy(label_smoothing=...) is unsupported on many
# Colab Keras builds. One-hot + categorical_crossentropy works everywhere.
def sparse_cce_with_smoothing(y_true, y_pred):
    smoothing = 0.08
    y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
    y_true = tf.one_hot(y_true, num_classes)
    return keras.losses.categorical_crossentropy(
        y_true, y_pred, label_smoothing=smoothing
    )

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=sparse_cce_with_smoothing,
    metrics=["accuracy"],
)
model.summary()


In [ ]:
# 10) Phase 1 — train head
BEST_MODEL_PATH = "/content/best_cardamom_grade_fixed.keras"

callbacks = [
    keras.callbacks.ModelCheckpoint(
        BEST_MODEL_PATH,
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=8,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
]

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    class_weight=class_weight,
    callbacks=callbacks,
)


In [ ]:
# 11) Phase 2 — fine-tune last EfficientNet blocks
base_model.trainable = True
# Freeze early layers; fine-tune deeper feature blocks
fine_tune_at = int(len(base_model.layers) * 0.70)
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=sparse_cce_with_smoothing,
    metrics=["accuracy"],
)

print(f"Fine-tuning from layer {fine_tune_at}/{len(base_model.layers)}")
trainable = sum(1 for l in base_model.layers if l.trainable)
print(f"Trainable base layers: {trainable}")

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weight,
    callbacks=callbacks,
)


In [ ]:
# 12) Evaluate best checkpoint
best_model = keras.models.load_model(
    BEST_MODEL_PATH,
    custom_objects={"sparse_cce_with_smoothing": sparse_cce_with_smoothing},
)
test_loss, test_acc = best_model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")


In [ ]:
# 13) Classification report + confusion matrix
y_true, y_pred = [], []
for images, labels in test_ds:
    probs = best_model.predict(images, verbose=0)
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(np.argmax(probs, axis=1).tolist())

print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, xticks_rotation=45, cmap=False)
plt.title("Confusion matrix (test)")
plt.tight_layout()
plt.show()

# Highlight Grade C mistakes (the failure mode you saw)
if "Cardmom Grade C" in class_names:
    c_idx = class_names.index("Cardmom Grade C")
    a_idx = (
        class_names.index("Cardmom Grade A")
        if "Cardmom Grade A" in class_names
        else None
    )
    c_total = cm[c_idx].sum()
    c_correct = cm[c_idx, c_idx]
    print(f"Grade C recall: {c_correct}/{c_total} = {c_correct / max(c_total, 1):.4f}")
    if a_idx is not None:
        print(f"Grade C predicted as A: {cm[c_idx, a_idx]}")


In [ ]:
# 14) Single-image helper (same crop as training)
def predict_single_image(image_path, model, threshold=0.70):
    img_bgr = cv2.imread(str(image_path))
    if img_bgr is None:
        return {"status": "error", "message": "Image not found or cannot read image."}

    cropped = crop_pod_to_square(img_bgr)
    if cropped is None:
        return {
            "status": "invalid",
            "message": "No clear cardamom pod detected. Use a plain background.",
        }

    img_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(img_rgb, IMG_SIZE)
    input_arr = np.expand_dims(resized.astype(np.float32), axis=0)

    probs = model.predict(input_arr, verbose=0)[0]
    pred_index = int(np.argmax(probs))
    confidence = float(probs[pred_index])
    result = "Uncertain" if confidence < threshold else class_names[pred_index]

    plt.figure(figsize=(5, 5))
    plt.imshow(resized.astype(np.uint8))
    plt.title(f"{result} | {confidence:.2f}")
    plt.axis("off")
    plt.show()

    return {
        "grade": result,
        "confidence": confidence,
        "all_probabilities": {
            class_names[i]: float(probs[i]) for i in range(len(class_names))
        },
    }

# Quick sanity check on a test image
sample_image = next((CROPPED_ROOT / "test").rglob("*.jpg"), None)
if sample_image is None:
    sample_image = next((CROPPED_ROOT / "test").rglob("*.jpeg"))
print("Sample:", sample_image)
print(predict_single_image(sample_image, best_model))


In [ ]:
# 15) Optional: upload a Grade C pod and verify
from google.colab import files

uploaded = files.upload()
for filename in uploaded.keys():
    print("\nTesting:", filename)
    result = predict_single_image(filename, best_model, threshold=0.70)
    if result.get("status") in {"error", "invalid"}:
        print(result)
    else:
        print("Predicted Grade:", result["grade"])
        print(f"Confidence: {result['confidence'] * 100:.2f}%")
        for name, p in result["all_probabilities"].items():
            print(f"  {name}: {p * 100:.2f}%")


In [ ]:
# 16) Save artifacts
CLASS_NAMES_PATH = "/content/class_names.json"
RESULTS_PATH = "/content/training_results.json"

with open(CLASS_NAMES_PATH, "w", encoding="utf-8") as f:
    json.dump(class_names, f, indent=2)

results = {
    "model_file": "best_cardamom_grade_fixed.keras",
    "architecture": "EfficientNetB0",
    "img_size": list(IMG_SIZE),
    "crop_pad_ratio": CROP_PAD_RATIO,
    "label_smoothing": 0.08,
    "class_names": class_names,
    "train_counts": dict(train_counts),
    "class_weight": {class_names[i]: class_weight[i] for i in range(num_classes)},
    "test_accuracy": float(test_acc),
    "test_loss": float(test_loss),
    "server_note": (
        "After deploy, set pad = int(max(w, h) * 0.25) + 20 in "
        "main.py detect_pod_crop_and_measure (must match CROP_PAD_RATIO)."
    ),
}
with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print("Saved:", BEST_MODEL_PATH)
print("Saved:", CLASS_NAMES_PATH)
print("Saved:", RESULTS_PATH)
print(json.dumps(results, indent=2))


In [ ]:
# 17) Download for your server
from google.colab import files

files.download(BEST_MODEL_PATH)
files.download(CLASS_NAMES_PATH)
files.download(RESULTS_PATH)

print("""
Deploy checklist:
1. Copy best_cardamom_grade_fixed.keras → models/grading/
2. Copy class_names.json → models/grading/
3. In main.py detect_pod_crop_and_measure, change:
     pad = int(max(w, h) * 0.8) + 20
   to:
     pad = int(max(w, h) * 0.25) + 20
4. Restart the API server
""")
